# Laboratorio 05: Transformada de Fourier Cuántica (QFT) de 3 Qubits

La QFT es el bloque fundamental de QPE, Shor y muchos algoritmos de estimación. Mapea la base computacional a la base de Fourier:
$$QFT|j\rangle = \frac{1}{\sqrt{N}}\sum_{k=0}^{N-1} e^{2\pi i jk/N}|k\rangle$$

**Objetivo:** construir la QFT desde cero, verificar unitariedad y visualizar el espectro.

**Prerequisito:** Módulos 01, 05 (algoritmos).

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT
from qiskit.quantum_info import Statevector, Operator
import numpy as np
import matplotlib.pyplot as plt
print('Entorno listo')

## 1. Construcción manual de la QFT

La QFT de n qubits aplica puertas Hadamard y rotaciones controladas R_k:
$$R_k = \begin{pmatrix}1&0\\0&e^{2\pi i/2^k}\end{pmatrix}$$

In [ ]:
def qft_manual(n: int) -> QuantumCircuit:
    'QFT de n qubits construida desde cero.'
    qc = QuantumCircuit(n, name=f'QFT_{n}')
    for j in range(n - 1, -1, -1):
        qc.h(j)
        for k in range(j - 1, -1, -1):
            angle = np.pi / (2 ** (j - k))
            qc.cp(angle, k, j)  # Rotacion de fase controlada
    # Swap final para corregir orden de bits
    for i in range(n // 2):
        qc.swap(i, n - 1 - i)
    return qc

n = 3
qc_qft = qft_manual(n)
print(f'QFT manual ({n} qubits):')
print(qc_qft.draw('text'))
print(f'  Puertas: {qc_qft.count_ops()}')
print(f'  Profundidad: {qc_qft.depth()}')

## 2. Verificar equivalencia con la librería de Qiskit

In [ ]:
qc_lib = QFT(n, do_swaps=True)
U_manual = Operator(qft_manual(n)).data
U_lib    = Operator(QFT(n, do_swaps=True)).data

# Verificar U_manual ~= U_lib (pueden diferir por fase global)
from qiskit.quantum_info import process_fidelity
diff = np.max(np.abs(np.abs(U_manual) - np.abs(U_lib)))
print(f'Diferencia maxima entre QFT manual y Qiskit: {diff:.2e}')
print(f'Equivalentes: {diff < 1e-10}')

# Verificar unitariedad: U†U = I
print(f'Unitariedad (U†U - I max): {np.max(np.abs(U_manual @ U_manual.conj().T - np.eye(2**n))):.2e}')

## 3. La QFT como transformada de frecuencias

In [ ]:
# Estado de ejemplo: superposicion de dos frecuencias
psi = np.zeros(2**n, dtype=complex)
psi[1] = 1/np.sqrt(2)  # frecuencia 1
psi[3] = 1/np.sqrt(2)  # frecuencia 3

qc_test = QuantumCircuit(n)
qc_test.initialize(psi, range(n))
qc_test.compose(qft_manual(n), inplace=True)
sv = Statevector(qc_test)
probs_qft = sv.probabilities()

# Comparar con FFT clasica
fft_clasica = np.abs(np.fft.fft(psi))**2 / 2**n

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bases = list(range(2**n))
axes[0].bar(bases, np.abs(psi)**2, color='steelblue', alpha=0.8, edgecolor='black')
axes[0].set_title('Estado de entrada |ψ⟩'); axes[0].set_xlabel('Estado base')
axes[0].set_ylabel('Probabilidad'); axes[0].set_xticks(bases)

axes[1].bar(bases, probs_qft, color='coral', alpha=0.8, edgecolor='black', label='QFT cuantica')
axes[1].plot(bases, fft_clasica, 'k--o', ms=8, lw=2, label='FFT clasica')
axes[1].set_title('QFT del estado'); axes[1].set_xlabel('Frecuencia k')
axes[1].set_ylabel('Amplitud^2'); axes[1].set_xticks(bases); axes[1].legend()
plt.tight_layout(); plt.savefig('qft_espectro.png', dpi=100); plt.show()
print('La QFT cuantica coincide exactamente con la FFT clasica.')

In [ ]:
# Comparativa de complejidad: QFT vs FFT clasica
ns = range(2, 16)
puertas_qft = [n*(n+1)//2 for n in ns]   # O(n^2) puertas
ops_fft = [n*(2**n) for n in ns]          # O(n*2^n) operaciones clasicas

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(ns, ops_fft, 'r-o', lw=2, label='FFT clasica: O(n·2^n)')
ax.semilogy(ns, puertas_qft, 'b-s', lw=2, label='QFT cuantica: O(n^2) puertas')
ax.set_xlabel('Numero de qubits n'); ax.set_ylabel('Operaciones (escala log)')
ax.set_title('QFT vs FFT clasica: escalado')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('qft_escalado.png', dpi=100); plt.show()